# Tutorial 3: Fitting the Top Quark Mass with in-situ JES calibration

In this tutorial we extract the top quark mass from the invariant mass distribution of the 3-jet combination (reconstructed top) while simultaneously fitting the Jet Energy Scale (JES) using the 2-jet combination (reconstructed W).

We use a 2D template approach ($m_W$ vs $m_t$) to capture the correlation between the two observables. The JES acts as a constrained shape systematic that shifts both $m_W$ and $m_t$. The top mass shift acts as an unconstrained parameter affecting only $m_t$.

## Setting up the environment

In [1]:
import os
import sys
from pathlib import Path

rabbit_base = str(Path(os.getcwd()).parent.absolute())
os.environ['RABBIT_BASE'] = rabbit_base

if rabbit_base not in sys.path:
    sys.path.append(rabbit_base)

rabbit_bin = os.path.join(rabbit_base, 'bin')
current_path = os.environ.get('PATH', '')
if rabbit_bin not in current_path:
    os.environ['PATH'] = f"{current_path}:{rabbit_bin}"

print("RABBIT_BASE:", os.environ['RABBIT_BASE'])
print("PATH:", os.environ['PATH'])

## Generating the invariant mass distribution (2D: $m_W$ vs $m_t$)

We will generate an invariant mass distribution representing the reconstructed W and top masses. 
We use pseudo-data and generate a signal peak on top of a background.

In [2]:
import numpy as np
import hist
import matplotlib.pyplot as plt
import mplhep as hep
from rabbit import tensorwriter
hep.style.use("CMS")

outdir = "results/top_mass"
os.makedirs(outdir, exist_ok=True)

w_mass_axis = hist.axis.Regular(20, 50, 110, name="m_W", label="Reconstructed W Mass [GeV]")
top_mass_axis = hist.axis.Regular(40, 130, 210, name="m_top", label="Reconstructed Top Mass [GeV]")

def get_hist(axes):
    return hist.Hist(*axes, storage=hist.storage.Weight())

m_top_nom = 172.5
m_w_nom = 80.4
dm_top = 1.0

jes_nom = 1.0
djes = 0.02

np.random.seed(42)
n_sig = 50000
n_bkg = 20000

bkg_w_mass = np.random.uniform(50, 110, n_bkg)
bkg_top_mass = np.random.uniform(130, 210, n_bkg)
h_bkg = get_hist((w_mass_axis, top_mass_axis))
h_bkg.fill(bkg_w_mass, bkg_top_mass)

def generate_signal(m_top, jes):
    w_res, top_res = 8.0, 15.0
    w_mass_base = np.random.normal(m_w_nom, w_res, n_sig)
    top_mass_base = np.random.normal(m_top, top_res, n_sig)
    w_mass_reco = w_mass_base * jes
    top_mass_reco = top_mass_base * jes
    h = get_hist((w_mass_axis, top_mass_axis))
    h.fill(w_mass_reco, top_mass_reco)
    return h

h_sig_nom = generate_signal(m_top_nom, jes_nom)
h_sig_mtop_up = generate_signal(m_top_nom + dm_top, jes_nom)
h_sig_mtop_dn = generate_signal(m_top_nom - dm_top, jes_nom)
h_sig_jes_up = generate_signal(m_top_nom, jes_nom + djes)
h_sig_jes_dn = generate_signal(m_top_nom, jes_nom - djes)

bkg_w_mass_up, bkg_top_mass_up = bkg_w_mass * (1+djes), bkg_top_mass * (1+djes)
bkg_w_mass_dn, bkg_top_mass_dn = bkg_w_mass * (1-djes), bkg_top_mass * (1-djes)
h_bkg_jes_up = get_hist((w_mass_axis, top_mass_axis))
h_bkg_jes_dn = get_hist((w_mass_axis, top_mass_axis))
h_bkg_jes_up.fill(bkg_w_mass_up, bkg_top_mass_up)
h_bkg_jes_dn.fill(bkg_w_mass_dn, bkg_top_mass_dn)

m_top_true = 173.2
jes_true = 1.01
h_sig_true = generate_signal(m_top_true, jes_true)
h_bkg_true = get_hist((w_mass_axis, top_mass_axis))
h_bkg_true.fill(bkg_w_mass * jes_true, bkg_top_mass * jes_true)
h_data = get_hist((w_mass_axis, top_mass_axis))
h_data.view()[...] = h_sig_true.view() + h_bkg_true.view()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
hep.histplot([h_bkg.project("m_W"), h_sig_nom.project("m_W")], stack=True, histtype='fill', label=['Background', 'Signal'], ax=ax1)
hep.histplot(h_data.project("m_W"), histtype='errorbar', color='black', label='Pseudodata', ax=ax1)
ax1.set_xlabel("Reconstructed W Mass [GeV]")
ax1.set_ylabel("Events")
ax1.legend()

hep.histplot([h_bkg.project("m_top"), h_sig_nom.project("m_top")], stack=True, histtype='fill', label=['Background', 'Signal'], ax=ax2)
hep.histplot(h_data.project("m_top"), histtype='errorbar', color='black', label='Pseudodata', ax=ax2)
ax2.set_xlabel("Reconstructed Top Mass [GeV]")
ax2.set_ylabel("Events")
ax2.legend()
plt.show()

## Writing the tensor

The channel is now a 2D histogram. The top mass dependence is encoded as an unconstrained shape variation (`top_mass_shift`). The JES is added as a standard shape systematic.

In [3]:
writer = tensorwriter.TensorWriter()

writer.add_channel(h_data.axes, "mass_region_2d")
writer.add_data(h_data, "mass_region_2d")

writer.add_process(h_sig_nom, "signal", "mass_region_2d", signal=True)
writer.add_process(h_bkg, "background", "mass_region_2d", signal=False)

writer.add_norm_systematic("bkg_norm", "background", "mass_region_2d", 1.10)

writer.add_systematic(
    [h_sig_mtop_up, h_sig_mtop_dn], 
    "top_mass_shift", 
    "signal", 
    "mass_region_2d", 
    symmetrize="average", 
    constrained=False,  
    noi=True            
)

writer.add_systematic(
    [h_sig_jes_up, h_sig_jes_dn], 
    "jes", 
    "signal", 
    "mass_region_2d", 
    symmetrize="average", 
    constrained=True
)

writer.add_systematic(
    [h_bkg_jes_up, h_bkg_jes_dn], 
    "jes", 
    "background", 
    "mass_region_2d", 
    symmetrize="average", 
    constrained=True
)

out_hdf5 = f"{outdir}/top_mass_jes_tensor.hdf5"
writer.write(outfolder=outdir, outfilename="top_mass_jes_tensor")
print(f"Tensor written to {out_hdf5}")

## Running the fit

We will scan the `top_mass_shift` parameter.

In [4]:
!rabbit_fit.py {outdir}/top_mass_jes_tensor.hdf5 -t 0 -o {outdir} --scan top_mass_shift --unblind --postfix mass_fit_jes

## Interpreting the result

The pulls and constraints. Notice how the JES is constrained to its injected value of +1% (+0.5 pull because $1\sigma$ is 2%).

In [5]:
!rabbit_print_pulls_and_constraints.py {outdir}/fitresults_mass_fit_jes.hdf5

## Plotting the likelihood scan

In [6]:
!rabbit_plot_likelihood_scan.py {outdir}/fitresults_mass_fit_jes.hdf5 --param top_mass_shift -o {outdir}

from IPython.display import HTML, display
import time
v = time.time()
display(HTML(
    f'<img src="{outdir}/nll_scan_top_mass_shift_data_unblinded.png?v={v}" style="width: 60%;" />'
))